# DICE ITC Results Notebook

This notebook is the public, portable entry point for reproducing the DICE ITC results. Run it top to bottom.

It regenerates:
- the core analysis tables and figures
- the full DICE paper results
- the workload-holdout appendix results
- the `results_itc_paper/` and `results_itc_appendix/` bundles
- the reproducibility manifest

Methodology reflected here:
1. Benign-only regime-conditioned micro-twin heads across Tier-0, Tier-0/1, and Tier-0/1/2 observation availability.
2. Online residualization with fixed block summaries.
3. Sequential conformal decisioning with persistent alerts.
4. Mechanism-level diagnosis from grouped residual evidence.
5. Workload-holdout robustness as a portable workload/software-drift proxy.
6. Reduced-observability robustness across tier subsets.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import tempfile

import pandas as pd
from IPython.display import Image, display

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)


In [ ]:
def resolve_repo_root(start: Path) -> Path:
    for base in [start, *start.parents]:
        if (base / 'tools' / 'run_results_pipeline.py').exists() and (base / 'data generation').exists():
            return base
    raise RuntimeError('Could not locate the DICE repository root from the current working directory.')


def portable_env() -> dict[str, str]:
    env = os.environ.copy()
    env['MPLCONFIGDIR'] = env.get('MPLCONFIGDIR', tempfile.mkdtemp(prefix='dice-mpl-'))
    env['MPLBACKEND'] = 'Agg'
    env['OPENBLAS_NUM_THREADS'] = '1'
    env['OMP_NUM_THREADS'] = '1'
    env['MKL_NUM_THREADS'] = '1'
    env['NUMEXPR_NUM_THREADS'] = '1'
    env['VECLIB_MAXIMUM_THREADS'] = '1'
    env['BLIS_NUM_THREADS'] = '1'
    env['PYTHONHASHSEED'] = '0'
    return env


REPO_ROOT = resolve_repo_root(Path.cwd().resolve())
DATASET_ROOT = REPO_ROOT / 'data generation' / 'dataset' / 'ITC_M2Pro_DATA'
_INTERNAL_PIPELINE = REPO_ROOT / 'tools' / 'run_results_pipeline.py'
OUT = DATASET_ROOT / 'results_analysis'
FIG = OUT / 'figures'
OUT_FULL = DATASET_ROOT / 'results_dice_full'
OUT_HOLDOUT = DATASET_ROOT / 'results_dice_full_holdout'
OUT_PAPER = DATASET_ROOT / 'results_itc_paper'
OUT_APPENDIX = DATASET_ROOT / 'results_itc_appendix'
MANIFEST = DATASET_ROOT / 'results_portable' / 'run_manifest.json'

print('REPO_ROOT    :', REPO_ROOT)
print('DATASET_ROOT :', DATASET_ROOT)


## Run End-to-End

Set `RUN_END_TO_END=True` and execute the next cell. The notebook will orchestrate the internal backend automatically and write the paper and appendix bundles.


In [ ]:
RUN_END_TO_END = True
INCLUDE_TUNING = False

if RUN_END_TO_END:
    cmd = [sys.executable, str(_INTERNAL_PIPELINE), '--package_itc']
    if INCLUDE_TUNING:
        cmd.append('--run_tuning')
    proc = subprocess.run(
        cmd,
        cwd=str(REPO_ROOT),
        env=portable_env(),
        capture_output=True,
        text=True,
    )
    print(proc.stdout)
    if proc.returncode != 0:
        print(proc.stderr)
        raise RuntimeError(f'Internal DICE pipeline failed with exit code {proc.returncode}')
else:
    print('Skipped end-to-end run. Set RUN_END_TO_END=True to execute.')


## Core Analysis Outputs

These are the tier-level analysis tables and figures used to describe separability and dataset coverage.


In [ ]:
overall = pd.read_csv(OUT / 'table_overall_metrics.csv')
stressor = pd.read_csv(OUT / 'table_stressor_metrics.csv')
workload = pd.read_csv(OUT / 'table_workload_summary.csv')
features = pd.read_csv(OUT / 'table_feature_inventory.csv')
quality = pd.read_csv(OUT / 'table_case_quality.csv')

print('Overall metrics')
display(overall)

print('Per-stressor metrics')
display(stressor)

print('Workload summary')
display(workload)

print('Feature inventory')
display(features[['tier_name', 'n_features_common', 'n_features_union']])

print('Case quality snapshot')
display(quality.head())


In [ ]:
for path in [
    FIG / 'fig_heatmap_pr_auc.png',
    FIG / 'fig_run_score_distributions.png',
    FIG / 'fig_af_timeseries_tier2.png',
]:
    print(path)
    if path.exists():
        display(Image(filename=str(path)))


## Methodology-Oriented Full Results

These outputs align with the preferred methodology: benign-only modeling, sequential decisioning, mechanism-level diagnosis, and robustness across observation heads.


In [ ]:
overall_full = pd.read_csv(OUT_FULL / 'overall_metrics.csv')
stressor_full = pd.read_csv(OUT_FULL / 'stressor_metrics_final_config.csv')
sequential = pd.read_csv(OUT_FULL / 'sequential_metrics.csv')
diagnosis = pd.read_csv(OUT_FULL / 'stressor_diagnosis_metrics.csv')
mechanism = pd.read_csv(OUT_FULL / 'mechanism_group_summary.csv')
tier_contrib = pd.read_csv(OUT_FULL / 'stressor_tier_contributions.csv')

print('Overall full-pipeline metrics')
display(overall_full)

print('Sequential decision metrics')
display(sequential)

print('Mechanism-group diagnosis metrics')
display(diagnosis)

print('Mechanism-group summary by stressor')
display(mechanism)

print('Tier contribution summary by stressor')
display(tier_contrib)

row_final = overall_full[overall_full['config'] == 'tier0_tier1_tier2'].iloc[0]
seq_final = sequential[sequential['config'] == 'tier0_tier1_tier2'].iloc[0]
diag_final = diagnosis[diagnosis['config'] == 'tier0_tier1_tier2'].iloc[0]
print('Final config (Tier-0 + Tier-1 + Tier-2)')
print('ROC-AUC              :', round(float(row_final['roc_auc_wc']), 4))
print('AUC-PR               :', round(float(row_final['pr_auc_wc']), 4))
print('Benign alert rate    :', round(float(seq_final['benign_run_alert_rate']), 4))
print('Median time-to-detect:', round(float(seq_final['median_time_to_detect_s']), 2))
print('Top-1 diagnosis acc  :', round(float(diag_final['top1_acc']), 4))
print('Top-2 diagnosis acc  :', round(float(diag_final['top2_acc']), 4))


In [ ]:
for path in [
    OUT_FULL / 'figures' / 'fig_roc_pr_by_config_wc.png',
    OUT_FULL / 'figures' / 'fig_detection_latency.png',
    OUT_FULL / 'figures' / 'fig_mechanism_group_summary.png',
    OUT_FULL / 'figures' / 'fig_stressor_confusion_matrix.png',
    OUT_FULL / 'figures' / 'fig_stressor_tier_contributions.png',
]:
    print(path)
    if path.exists():
        display(Image(filename=str(path)))


## Workload/Software Drift Proxy and Appendix Bundles

`results_dice_full_holdout/` is the portable workload-holdout evaluation. In this notebook, it is the practical proxy for workload/software drift robustness.


In [ ]:
if (OUT_HOLDOUT / 'holdout_robustness_summary.csv').exists():
    holdout = pd.read_csv(OUT_HOLDOUT / 'holdout_robustness_summary.csv')
    print('Holdout robustness summary')
    display(holdout)
else:
    print('No holdout summary found yet.')

for folder in [OUT_PAPER, OUT_APPENDIX]:
    print('
', folder)
    if folder.exists():
        files = sorted(str(p.relative_to(folder)) for p in folder.rglob('*') if p.is_file())
        display(pd.DataFrame({'file': files[:100]}))
    else:
        print('Missing:', folder)


In [ ]:
manifest = json.loads(MANIFEST.read_text())
print('Manifest path:', MANIFEST)
print('Dataset SHA256:', manifest['dataset_digest']['sha256'])
print('Environment file SHA256:', manifest['environment_files']['environment_yml']['sha256'])
print('Requirements SHA256:', manifest['environment_files']['requirements_txt']['sha256'])
